In [1]:
import pandas as pd

In [2]:
import pyterrier as pt
from pyterrier_quality import QualT5

qt5 = QualT5('pyterrier-quality/qt5-small')

/opt/miniconda3/envs/doc_quality/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
if not pt.java.started():
    pt.java.init()

Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


In [4]:
index_path ='/mnt/indices/msmarco-passage.terrier/'
index_ref = pt.IndexRef.of(index_path)
index = pt.IndexFactory.of(index_ref)

14:21:35.006 [main] WARN org.terrier.structures.BaseCompressingMetaIndex -- Structure meta reading data file directly from disk (SLOW) - try index.meta.data-source=fileinmem in the index properties file. 1.9 GiB of memory would be required.


In [5]:
res_name = 'e5_dev_small'
retr_res = pd.read_csv(f'../../rag_utility/res/{res_name}.csv')

In [6]:
text_loader = index.text_loader(["text"])

In [24]:
import numpy as np
import pathlib
from tqdm import tqdm

_k = 3
output_filename = f"./quality_res/{res_name}_integrated_{_k}.csv"
csvfile = pathlib.Path(output_filename)
if(csvfile.exists()):
    exist_output = pd.read_csv(output_filename)
    exist_qids = exist_output.qid.unique()
    del exist_output
else:
    exist_qids = []

df_content = []
for _qid in tqdm(retr_res.qid.unique()):
    if(_qid in exist_qids):
        continue
    csvfile = pathlib.Path(output_filename)
    integrated_text = ''
    for _t, _i in zip(text_loader(retr_res[(retr_res.qid==_qid)&(retr_res['rank']<_k)])['text'], range(_k)):
        integrated_text += f'Context {_i+1}: "{_t}";\n'
        
    df_content.append([_qid, retr_res[(retr_res.qid==_qid)]['query'].values[0], f'{_qid}_itg_{_k}', integrated_text])

    if((len(df_content)==10)|(_qid==retr_res.qid.unique()[-1])):
        df_for_esti = pd.DataFrame(df_content, columns=['qid', 'query', 'docno', 'text'])
        temp_output = qt5(df_for_esti)
        temp_output.to_csv(output_filename, mode='a', index=False, header=not csvfile.exists())
        df_content = []

 26%|██▌       | 1789/6980 [25:15<1:13:18,  1.18it/s]


KeyboardInterrupt: 